In [ ]:
import pandas as pd
import numpy as np
import os
import statsmodels.formula.api as smf

# Set base directory relative to notebook location
BASE_DIR = os.path.dirname(os.path.abspath('03_demand_model.ipynb'))
RAW_DIR = os.path.join(BASE_DIR, 'data', 'raw')
CLEAN_DIR = os.path.join(BASE_DIR, 'data', 'cleaned')
FIG_DIR = os.path.join(BASE_DIR, 'outputs', 'figures')

# Load registration data
df_reg = pd.read_csv(os.path.join(CLEAN_DIR, 'ev_registrations_clean.csv'))
df_reg['quarter'] = pd.PeriodIndex(df_reg['quarter'], freq='Q')
print("Registration data:", df_reg.shape)
print(df_reg.head())

In [ ]:
import pandas as pd
import numpy as np
import os

# Load Google Trends - resample to quarterly
df_trends = pd.read_csv(os.path.join(RAW_DIR, 'trends_competition.csv'), index_col=0, parse_dates=True)
df_trends_q = df_trends.resample('QE').mean()
df_trends_q.index = df_trends_q.index.to_period('Q')
df_trends_q = df_trends_q.reset_index()
df_trends_q.columns = ['quarter', 'tesla_search', 'byd_search', 'nio_search', 'rivian_search', 'lucid_search']

# Load gas prices - filter to US average, resample to quarterly
df_gas = pd.read_csv(os.path.join(RAW_DIR, 'eia_gas_prices.csv'))
df_gas = df_gas[df_gas['product-name'] == 'Total Gasoline'].copy()
df_gas['period'] = pd.to_datetime(df_gas['period'])
df_gas['quarter'] = df_gas['period'].dt.to_period('Q')
df_gas_q = df_gas.groupby('quarter')['value'].mean().reset_index()
df_gas_q.columns = ['quarter', 'gas_price']

# Load Supercharger data - count by state
df_sc = pd.read_csv(os.path.join(RAW_DIR, 'afdc_superchargers.csv'))
df_sc['open_date'] = pd.to_datetime(df_sc['open_date'], errors='coerce')
df_sc = df_sc.dropna(subset=['open_date', 'state'])

# Count cumulative superchargers per state per quarter
df_sc['quarter'] = df_sc['open_date'].dt.to_period('Q')
sc_counts = df_sc.groupby(['state', 'quarter']).size().reset_index(name='new_chargers')
sc_counts = sc_counts.sort_values(['state', 'quarter'])
sc_counts['cumulative_chargers'] = sc_counts.groupby('state')['new_chargers'].cumsum()

# Filter registration data to 2020 onwards
df_reg = df_reg[df_reg['quarter'].astype(str) >= '2020Q1'].copy()

# Merge everything
df_model = df_reg.merge(df_trends_q, on='quarter', how='left')
df_model = df_model.merge(df_gas_q, on='quarter', how='left')
df_model = df_model.merge(sc_counts[['state', 'quarter', 'cumulative_chargers']], 
                           on=['state', 'quarter'], how='left')

# Add policy dummies
df_model['post_ira'] = (df_model['quarter'].astype(str) >= '2022Q3').astype(int)
df_model['post_tariff'] = (df_model['quarter'].astype(str) >= '2024Q2').astype(int)

# Fill missing charger counts with 0
df_model['cumulative_chargers'] = df_model['cumulative_chargers'].fillna(0)

print("Model dataset shape:", df_model.shape)
print("\nMissing values:\n", df_model.isnull().sum())
print("\nSample:\n", df_model.head())

In [ ]:
# Fix 1: Drop rows missing tesla_share
df_model = df_model.dropna(subset=['tesla_share'])

# Fix 2: Fill missing Trends with forward fill then backfill
df_model = df_model.sort_values(['state', 'quarter'])
trend_cols = ['tesla_search', 'byd_search', 'rivian_search', 'lucid_search']
df_model[trend_cols] = df_model[trend_cols].fillna(method='ffill').fillna(method='bfill')

# Fix 3: Gas prices - use national average (area-name = 'U.S.')
df_gas2 = pd.read_csv(os.path.join(RAW_DIR, 'eia_gas_prices.csv'))
df_gas2 = df_gas2[
    (df_gas2['product-name'] == 'Total Gasoline') & 
    (df_gas2['area-name'].str.contains('U.S.', na=False))
].copy()
df_gas2['period'] = pd.to_datetime(df_gas2['period'])
df_gas2['quarter'] = df_gas2['period'].dt.to_period('Q')
df_gas_national = df_gas2.groupby('quarter')['value'].mean().reset_index()
df_gas_national.columns = ['quarter', 'gas_price_national']

# Merge national gas price
df_model = df_model.drop(columns='gas_price')
df_model = df_model.merge(df_gas_national, on='quarter', how='left')

print("Missing values after fixes:")
print(df_model.isnull().sum())
print("\nShape:", df_model.shape)

In [ ]:
# Check what area names exist in gas data
df_gas_check = pd.read_csv(os.path.join(RAW_DIR, 'eia_gas_prices.csv'))
print("Unique area names:")
print(df_gas_check['area-name'].value_counts().head(20))

In [ ]:
# Fix gas prices - use U.S. national series
df_gas_fix = pd.read_csv(os.path.join(RAW_DIR, 'eia_gas_prices.csv'))
df_gas_fix = df_gas_fix[
    (df_gas_fix['area-name'] == 'U.S.') & 
    (df_gas_fix['product-name'] == 'Total Gasoline')
].copy()
df_gas_fix['quarter'] = pd.to_datetime(df_gas_fix['period']).dt.to_period('Q')
df_gas_national = df_gas_fix.groupby('quarter')['value'].mean().reset_index()
df_gas_national.columns = ['quarter', 'gas_price_national']

print("Gas price quarters sample:", df_gas_national['quarter'].dtype)
print("Model quarters sample:", df_model['quarter'].dtype)
print(df_gas_national.head())

# Drop old gas column if exists and merge fresh
if 'gas_price_national' in df_model.columns:
    df_model = df_model.drop(columns='gas_price_national')

df_model = df_model.merge(df_gas_national, on='quarter', how='left')

# Fix nio_search - just fill with 0 since it's negligible
df_model['nio_search'] = df_model['nio_search'].fillna(0)

print("\nMissing values after final fixes:")
print(df_model.isnull().sum())
print("Shape:", df_model.shape)

In [ ]:
# Force string conversion on both sides before merging
df_model['quarter_str'] = df_model['quarter'].astype(str)
df_gas_national['quarter_str'] = df_gas_national['quarter'].astype(str)

# Drop old gas column
if 'gas_price_national' in df_model.columns:
    df_model = df_model.drop(columns='gas_price_national')

# Merge on string version
df_model = df_model.merge(df_gas_national[['quarter_str', 'gas_price_national']], 
                           on='quarter_str', how='left')

print("Missing gas prices:", df_model['gas_price_national'].isnull().sum())
print("Sample:")
print(df_model[['state', 'quarter', 'gas_price_national']].head(10))

In [ ]:
# Sort and interpolate missing gas prices
df_model = df_model.sort_values(['state', 'quarter'])

# Fill gas price gaps using interpolation across the full time series
df_gas_filled = df_model[['quarter_str', 'gas_price_national']].drop_duplicates('quarter_str').sort_values('quarter_str')
df_gas_filled['gas_price_national'] = df_gas_filled['gas_price_national'].interpolate(method='linear')

# Drop old and merge filled version
df_model = df_model.drop(columns='gas_price_national')
df_model = df_model.merge(df_gas_filled[['quarter_str', 'gas_price_national']], 
                           on='quarter_str', how='left')

print("Missing gas prices:", df_model['gas_price_national'].isnull().sum())
print("\nGas price sample:")
print(df_model[['quarter', 'gas_price_national']].drop_duplicates().head(12))

In [ ]:
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

# Save clean model dataset
df_model.to_csv(os.path.join(CLEAN_DIR, 'model_dataset.csv'), index=False)

# Run OLS regression
# Dependent variable: tesla_share
# Independent variables: gas price, tesla search, rivian search, 
#                        supercharger density, post-IRA dummy, post-tariff dummy

formula = '''tesla_share ~ gas_price_national + tesla_search + rivian_search + 
             cumulative_chargers + post_ira + post_tariff'''

model = smf.ols(formula, data=df_model).fit()
print(model.summary())

In [ ]:
# Cleaner model without chargers
formula2 = '''tesla_share ~ gas_price_national + rivian_search + 
              post_ira + post_tariff'''

model2 = smf.ols(formula2, data=df_model).fit()
print(model2.summary())

Three significant findings:

Gas price (coef: -0.099, p=0.018) — A $1 increase in gas prices is associated with a ~10 percentage point drop in Tesla's market share. This reflects competitors capturing gas-price-driven EV converts more than Tesla does.
Post-IRA (coef: -0.059, p=0.009) — After the IRA passed, Tesla's share fell ~6 points. The policy grew the overall market but Tesla didn't proportionally benefit — other brands absorbed the new demand.
Post-tariff (coef: -0.089, p=0.000) — Strongest and most significant result. Tesla's share dropped ~9 points in the post-tariff period. This tells you the tariff didn't reverse Tesla's competitive erosion — it may have slowed Chinese entry but domestic competition continued taking share.

R² = 0.197 — your model explains ~20% of Tesla share variance, which is honest and defensible for a cross-state panel without fixed effects.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Coefficient plot
coefs = model2.params.drop('Intercept')
errors = model2.conf_int().drop('Intercept')
errors.columns = ['lower', 'upper']

fig, ax = plt.subplots(figsize=(10, 5))

colors = ['tomato' if p < 0.05 else 'steelblue' 
          for p in model2.pvalues.drop('Intercept')]

ax.barh(coefs.index, coefs.values, 
        xerr=[coefs.values - errors['lower'], errors['upper'] - coefs.values],
        color=colors, alpha=0.8, capsize=4)

ax.axvline(x=0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Coefficient (effect on Tesla market share)')
ax.set_title('OLS Regression: Drivers of Tesla Market Share\n(Red = statistically significant at p<0.05)')

labels = {
    'gas_price_national': 'Gas Price ($/gal)',
    'rivian_search': 'Rivian Search Interest',
    'post_ira': 'Post-IRA (2022Q3+)',
    'post_tariff': 'Post-Tariff (2024Q2+)'
}
ax.set_yticklabels([labels.get(t.get_text(), t.get_text()) 
                    for t in ax.get_yticklabels()])

plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'regression_coefficients.png'), dpi=150)
plt.show()

In [ ]:
formula3 = '''tesla_share ~ gas_price_national + rivian_search + 
              post_ira + post_tariff + C(state)'''
model3 = smf.ols(formula3, data=df_model).fit()
print(model3.rsquared)

In [ ]:
formula4 = '''tesla_share ~ gas_price_national + rivian_search + 
              post_ira + post_tariff + C(state) + C(quarter_str)'''

In [ ]:
model4 = smf.ols(formula4, data=df_model).fit()
print(f"R-squared: {model4.rsquared:.3f}")
print(f"Adj R-squared: {model4.rsquared_adj:.3f}")
print("\nKey coefficients (excluding state and quarter dummies):")

# Print only the variables we care about, not all the state dummies
key_vars = ['gas_price_national', 'rivian_search', 'post_ira', 'post_tariff']
for var in key_vars:
    coef = model4.params[var]
    pval = model4.pvalues[var]
    sig = "***" if pval < 0.01 else "**" if pval < 0.05 else "*" if pval < 0.1 else ""
    print(f"  {var:25s}  coef: {coef:+.4f}  p: {pval:.3f} {sig}")

In [ ]:
# Load combined panel
df_combined = pd.read_csv(os.path.join(CLEAN_DIR, 'ev_registrations_with_ca.csv'))
df_combined['quarter'] = pd.PeriodIndex(df_combined['quarter'], freq='Q')

# Filter to 2020 onwards
df_combined = df_combined[df_combined['quarter'].astype(str) >= '2020Q1'].copy()
df_combined['quarter_str'] = df_combined['quarter'].astype(str)

# Merge model variables back in
df_model_ca = df_combined.merge(df_trends_q, on='quarter', how='left')
df_model_ca = df_model_ca.merge(df_gas_national[['quarter_str', 'gas_price_national']], on='quarter_str', how='left')
df_model_ca['post_ira'] = (df_model_ca['quarter_str'] >= '2022Q3').astype(int)
df_model_ca['post_tariff'] = (df_model_ca['quarter_str'] >= '2024Q2').astype(int)
df_model_ca['nio_search'] = df_model_ca['nio_search'].fillna(0)
df_model_ca['gas_price_national'] = df_model_ca['gas_price_national'].fillna(method='ffill').fillna(method='bfill')

# Re-run two-way fixed effects model
formula_ca = '''tesla_share ~ gas_price_national + rivian_search + 
                post_ira + post_tariff + C(state) + C(quarter_str)'''

model_ca = smf.ols(formula_ca, data=df_model_ca).fit()

print(f"R-squared: {model_ca.rsquared:.3f}")
print(f"Adj R-squared: {model_ca.rsquared_adj:.3f}")
print(f"N observations: {model_ca.nobs:.0f}")
print("\nKey coefficients:")
key_vars = ['gas_price_national', 'rivian_search', 'post_ira', 'post_tariff']
for var in key_vars:
    coef = model_ca.params[var]
    pval = model_ca.pvalues[var]
    sig = "***" if pval < 0.01 else "**" if pval < 0.05 else "*" if pval < 0.1 else ""
    print(f"  {var:25s}  coef: {coef:+.4f}  p: {pval:.3f} {sig}")